# SI Table S6: per-solvent implicit vs explicit fit RMSE (chloroform reference, ¹H)

Per solvent (chloroform reference), fit RMSE of experimental solvent-induced ¹H shifts using implicit
(PCM) vs explicit (Desmond), and the percent improvement.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import pandas as pd

import delta22
import paths

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def document_path(name):
    os.makedirs("documents", exist_ok=True)
    return os.path.join("documents", name)

In [ ]:
# Figure 4 uses one DFT method for the proton sites; differences are taken against a reference
# solvent, and the solvent_mean pseudo-solvent is added for the solvent-averaged reference.
METHOD, BASIS, GEOM = "b3lyp_d3bj", "pcSseg2", "aimnet2"
q = delta22.load_query_df_dft(DELTA22_HDF5, XLSX, verbose=False)
one = q[(q["sap_nmr_method"] == METHOD) & (q["sap_basis"] == BASIS) & (q["sap_geometry_type"] == GEOM)]
one = delta22.add_solvent_mean(one)
print(len(one), "rows for", METHOD, BASIS, GEOM)

## Table S6: per-solvent implicit vs explicit fit RMSE (chloroform reference, 1H)

In [ ]:
rows = []
for solvent in ['tetrahydrofuran', 'dichloromethane', 'acetone', 'acetonitrile', 'dimethylsulfoxide', 'trifluoroethanol', 'methanol', 'TIP4P', 'benzene', 'toluene', 'chlorobenzene']:
    sp = delta22.solvent_pair_differences(one, solvent, "chloroform", nucleus="H", explicit="desmond")
    implicit = delta22.fit_differences_to_experimental(sp, "implicit_diff")["rmse"]
    explicit = delta22.fit_differences_to_experimental(sp, "explicit_diff")["rmse"]
    rows.append({"solvent": solvent,
                 "implicit_fitting_rmse": round(implicit, 4),
                 "explicit_fitting_rmse": round(explicit, 4),
                 "explicit_improvement_pct": round(100.0 * (implicit - explicit) / implicit, 2)})
table_s6 = pd.DataFrame(rows)
print(table_s6.to_string(index=False))

# write the table to this notebook's documents/ folder
out = document_path("si_table_s06_solvent_corrections.xlsx")
with pd.ExcelWriter(out) as writer:
    table_s6.to_excel(writer, sheet_name="Table S6", index=False)
print("wrote", os.path.relpath(out, REPO))